# 01 - EDA de Combinaciones de Componentes

Este notebook realiza la exploración específica de la columna `Composition`
**antes de limpiar**, con el objetivo de entender su estructura, detectar
anomalías y justificar las decisiones de limpieza implementadas en `cleaning.py`.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from src.load_data import load_medicine_data
from src.enfoque_01_combinaciones_componentes.cleaning import extraer_componentes

df = load_medicine_data(download_if_missing=False)
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

Dataset cargado: 11,825 filas × 9 columnas


## 1. Estructura general de la columna `Composition`

Antes de extraer componentes, inspeccionamos el formato raw de la columna
para entender con qué tipo de datos trabajaremos.

In [2]:
# Primeras filas de Composition
df["Composition"].head(10).tolist()

['Bevacizumab (400mg)',
 'Amoxycillin  (500mg) +  Clavulanic Acid (125mg)',
 'Azithromycin (500mg)',
 'Ambroxol (30mg/5ml) + Levosalbutamol (1mg/5ml) + Guaifenesin (50mg/5ml)',
 'Ranitidine (150mg)',
 'Fexofenadine (120mg)',
 'Pheniramine (25mg)',
 'Donepezil (5mg)',
 'Amoxycillin  (500mg) +  Clavulanic Acid (125mg)',
 'Hydroxyzine (25mg)']

## 2. Distribución por número de componentes (datos raw)

Contamos cuántos componentes tiene cada medicamento antes de cualquier limpieza,
usando `extraer_componentes` de forma exploratoria.

In [3]:
# Distribución de n_components en datos raw
df["n_components_preview"] = df["Composition"].apply(
    lambda x: len(extraer_componentes(x))
)

dist = df["n_components_preview"].value_counts().sort_index()
dist_pct = (dist / len(df) * 100).round(2)

resumen = pd.DataFrame({
    "Medicamentos": dist,
    "Porcentaje (%)": dist_pct
})
resumen.index.name = "n_componentes"
resumen

,Medicamentos,Porcentaje (%)
n_componentes,,
1,7069,59.78
2,3596,30.41
3,933,7.89
4,150,1.27
5,51,0.43
6,16,0.14
7,7,0.06
8,2,0.02
9,1,0.01


## 3. Ejemplos de cada tamaño de combinación

Inspeccionamos ejemplos reales para entender la estructura del dato
en cada categoría.

In [4]:
# Un ejemplo de cada tamaño
for n in sorted(df["n_components_preview"].unique()):
    ejemplo = df[df["n_components_preview"] == n]["Composition"].iloc[0]
    print(f"n={n}: {ejemplo}")

n=1: Bevacizumab (400mg)
n=2: Amoxycillin  (500mg) +  Clavulanic Acid (125mg)
n=3: Ambroxol (30mg/5ml) + Levosalbutamol (1mg/5ml) + Guaifenesin (50mg/5ml)
n=4: Camphor (0.01% w/v) + Menthol (0.005% w/v) + Naphazoline (0.05% w/v) + Phenylephrine (0.12% w/v)
n=5: Chlorpheniramine Maleate (0.5mg/5ml) + Paracetamol (125mg/5ml) + Phenylephrine (5mg/5ml) + Sodium Citrate (60mg/5ml) + Menthol (1mg/5ml)
n=6: Tranexamic Acid (10% w/w) + Kojic Acid (2% w/w) + Arbutin (1.5% w/w) + Magnesium (1% w/w) + Vitamin E (1% w/w) + Mulberry extract (1% w/w)
n=7: Calcitriol (0.25mcg) + Calcium citrate (1000mg) + Zinc Sulphate Monohydrate (7.5mg) + Methylcobalamin (1500mcg) + L-Methyl Folate (800mcg) + Boron (5mg) + Magnesium Oxide (50mg)
n=8: Levo-carnitine (1.5gm) + Fructo Oligosaccharide (1gm) + Coenzyme Q10 (100mg) + Citric Acid (50mg) + Folic Acid (500mcg) + Methylcobalamin (2.5mcg) + Selenium (50mcg) + Zinc (10mg)
n=9: Methylcobalamin (1500mcg) + Folic Acid (1.5mg) + Vitamin B6 (Pyridoxine) (3mg) + Vit

## 4. Detección de duplicados completos

Identificamos registros donde todas las columnas son idénticas.
Estos serán eliminados en el pipeline de limpieza.

In [5]:
duplicados = df.duplicated().sum()
print(f"Duplicados completos detectados: {duplicados}")
print(f"Porcentaje sobre el total: {duplicados / len(df) * 100:.2f}%")

# Ver ejemplos de duplicados
df[df.duplicated(keep=False)].sort_values("Composition").head(6)

Duplicados completos detectados: 84
Porcentaje sobre el total: 0.71%


,Medicine Name,Composition,Uses,Side_effects,Image URL,Manufacturer,Excellent Review %,Average Review %,Poor Review %,n_components_preview
2305,Excex Gel,Adapalene (0.1% w/w) + Clindamycin (1% w/w),Treatment of Acne,Skin peeling Erythema skin redness Itching Dry...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Zee Laboratories,51,41,8,2
2304,Excex Gel,Adapalene (0.1% w/w) + Clindamycin (1% w/w),Treatment of Acne,Skin peeling Erythema skin redness Itching Dry...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Zee Laboratories,51,41,8,2
6233,Noworm Chewable Tablet,Albendazole (400mg),Treatment of Parasitic infections,Vomiting Dizziness Increased liver enzymes Nau...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Alkem Laboratories Ltd,40,34,26,1
6209,Noworm Chewable Tablet,Albendazole (400mg),Treatment of Parasitic infections,Vomiting Dizziness Increased liver enzymes Nau...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Alkem Laboratories Ltd,40,34,26,1
11406,Xrate Cough Expectorant Sugar Free,Ambroxol (15mg/5ml) + Guaifenesin (50mg/5ml) +...,Cough,Nausea Diarrhea Vomiting Dizziness Headache Ra...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Celsius Healthcare Pvt Ltd,67,33,0,4
11405,Xrate Cough Expectorant Sugar Free,Ambroxol (15mg/5ml) + Guaifenesin (50mg/5ml) +...,Cough,Nausea Diarrhea Vomiting Dizziness Headache Ra...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Celsius Healthcare Pvt Ltd,67,33,0,4


## 5. Detección de anomalías en el formato de `Composition`

Buscamos valores que podrían causar problemas en el parseo posterior:
strings muy cortos, sin paréntesis, o con formatos inesperados.

In [6]:
# Composiciones que no tienen paréntesis (sin dosis explícita)
sin_parentesis = df[~df["Composition"].str.contains(r"\(", regex=True)]
print(f"Composiciones sin paréntesis: {len(sin_parentesis)}")
sin_parentesis["Composition"].head(5).tolist()

Composiciones sin paréntesis: 0


[]

## 6. Composiciones más frecuentes

Identificamos qué combinaciones se repiten más bajo distintos nombres
comerciales (sinónimos farmacéuticos).

In [7]:
# Normalización básica para agrupar (sin limpiar dosis)
df["comp_normalizada"] = (
    df["Composition"]
    .str.lower()
    .str.strip()
    .str.split("+")
    .apply(lambda x: " + ".join(sorted([i.strip() for i in x])))
)

top_composiciones = (
    df.groupby("comp_normalizada")
    .agg(
        n_medicamentos=("Medicine Name", "count"),
        n_nombres_distintos=("Medicine Name", "nunique")
    )
    .sort_values("n_medicamentos", ascending=False)
    .head(10)
    .reset_index()
)
top_composiciones

,comp_normalizada,n_medicamentos,n_nombres_distintos
0,luliconazole (1% w/w),98,64
1,levocetirizine (5mg) + montelukast (10mg),77,76
2,ketoconazole (2% w/w),66,58
3,domperidone (30mg) + rabeprazole (20mg),60,59
4,itraconazole (100mg),53,48
5,itraconazole (200mg),52,47
6,telmisartan (40mg),52,51
7,domperidone (30mg) + pantoprazole (40mg),47,47
8,amlodipine (5mg) + telmisartan (40mg),44,43
9,metformin (500mg),44,42


## 7. Conclusiones del EDA

A partir de este análisis se tomaron las siguientes decisiones de limpieza,
documentadas en detalle en `docs/decisiones_limpieza.md`:

1. **Eliminar 84 duplicados completos** — no aportan información nueva.
2. **Normalizar espacios en `Composition`** — garantiza parseo confiable del separador `+`.
3. **Extraer componentes sin dosis** — las dosis no son relevantes para analizar combinaciones.
4. **Generar `size_category` como Categorical ordenado** — para respetar el orden lógico en análisis posteriores.
5. **Marcar anomalías sin eliminar** — 0 anomalías detectadas; decisión documentada explícitamente.